# CU7 Performance Assessment — LangganKu (starter)
Fill in your name and the last four digits of your IC, then work through the Tasks of the paper **in this notebook**.
First create your own data file in a terminal: `python make_my_data.py <L4>`  (the file `subscriber_churn.csv` appears in this folder).
Keep the notebook runnable from top to bottom.

In [3]:
import warnings; warnings.filterwarnings("ignore")
import mlflow, numpy as np, pandas as pd

NAME = "PETER LING HUA ZHONG"      # e.g. "AISYAH BINTI AHMAD"
L4   = "0161"                # last four digits of your IC, keep the leading zero

# ---- your personal values (rules in the paper) ----
SEED      = int(64)
TEST_SIZE = 0.25
DEPTH     = 6
C1        = 0.16
C2        =  1.6
RANK      = 'test_recall'
PORT      =  5100 + int(L4[2:])
EXP       = 'LangganKu_Churn_0161'
REG       = 'sqlite:///langganku_0161.db'
print(dict(SEED=SEED, TEST_SIZE=TEST_SIZE, DEPTH=DEPTH, C1=C1,C2=C2, RANK =RANK, PORT=PORT, EXP=EXP, REG=REG, ))

{'SEED': 64, 'TEST_SIZE': 0.25, 'DEPTH': 6, 'C1': 0.16, 'C2': 1.6, 'RANK': 'test_recall', 'PORT': 5161, 'EXP': 'LangganKu_Churn_0161', 'REG': 'sqlite:///langganku_0161.db'}


## Task 1 — Set-up, split and pipeline

In [4]:
mlflow.set_tracking_uri(REG)
mlflow.set_experiment(EXP)

2026/10/20 09:24:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/20 09:24:17 INFO mlflow.store.db.utils: Updating database tables


2026/10/20 09:24:20 INFO mlflow.tracking.fluent: Experiment with name 'LangganKu_Churn_0161' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:C:/Users/Lenovo/Documents/PETER LING HUA ZHONG CU7/mlruns/1', creation_time=1792459460582, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792459460582, lifecycle_stage='active', name='LangganKu_Churn_0161', tags={}, trace_location=None, workspace='default'>

In [6]:
df = pd.read_csv("subscriber_churn.csv")
print("Shape:", df.shape)
print("\nColumn types:"); print(df.dtypes)
print("\nMissing values per column:"); print(df.isna().sum())
print("\nShare of each class in defaulted:")

display(df.head())

Shape: (1200, 10)

Column types:
subscriber_id      object
avg_hours_week    float64
payment_method     object
tenure_months       int64
support_calls       int64
plan               object
main_device        object
contract           object
monthly_fee_rm    float64
churned             int64
dtype: object

Missing values per column:
subscriber_id     0
avg_hours_week    0
payment_method    0
tenure_months     0
support_calls     0
plan              0
main_device       0
contract          0
monthly_fee_rm    0
churned           0
dtype: int64

Share of each class in defaulted:


,subscriber_id,avg_hours_week,payment_method,tenure_months,support_calls,plan,main_device,contract,monthly_fee_rm,churned
0,S0001,4.3,Bank transfer,8,1,Basic,Tablet,Monthly,26.7,0
1,S0003,23.2,Bank transfer,19,2,Premium,Laptop,Monthly,60.2,0
2,S0005,15.9,Cash voucher,13,1,Standard,Phone,Monthly,40.0,1
3,S0006,4.9,E-wallet,16,2,Basic,Phone,Monthly,25.2,1
4,S0007,10.8,Card,27,1,Basic,Phone,Yearly,26.4,0


In [7]:
X = df.drop(columns=["subscriber_id", "churned"])
y = df["churned"]

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

def make_pipeline(model):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ])
    return Pipeline([("prep", pre), ("model", model)])

In [10]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

def get_metrics(model, X, y):
    pred = model.predict(X)
    return {
        "rmse": root_mean_squared_error(y, pred),
        "mae": mean_absolute_error(y, pred),
        "r2": r2_score(y, pred),
    }

pipe = make_pipeline(LinearRegression())
pipe.fit(X_train, y_train)

get_metrics(pipe, X_test, y_test)

{'rmse': 0.36721331433979676,
 'mae': 0.29842029311826296,
 'r2': 0.28082336944843544}

## Task 2 — Track the candidates in ONE experiment

In [13]:
# your code
def log_candidate(run_name, model, model_type, params):
    pipe = make_pipeline(model)
    with mlflow.start_run(run_name=f"{run_name}-{L4}"):
        pipe.fit(X_train, y_train)

        mlflow.log_param("model", type(model).__name__)
        mlflow.log_param("test_size", TEST_SIZE)
        mlflow.log_param("split_seed", SEED)
        mlflow.log_params(params)

        mlflow.set_tag("model_type", model_type)

        for prefix, X_, y_ in [("train", X_train, y_train), ("test", X_test, y_test)]:
            for k, v in get_metrics(pipe, X_, y_).items():
                mlflow.log_metric(f"{prefix}_{k}", v)

        mlflow.sklearn.log_model(
            pipe,
            name="model",
            input_example=X_train.head(5),
            serialization_format="cloudpickle",
        )

In [14]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

log_candidate("linear-base", LinearRegression(),
              "linear", {})

log_candidate(f"ridge-a{C1:g}", Ridge(alpha=C1),
              "linear", {"alpha": C1})

log_candidate(f"ridge-a{C2:g}", Ridge(alpha=C2),
              "linear", {"alpha": C2})

log_candidate(f"rf-d{DEPTH}", RandomForestRegressor(n_estimators=100, max_depth=DEPTH, random_state=SEED),
              "tree_ensemble", {"n_estimators": 100, "max_depth": DEPTH})

2026/10/20 09:41:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/20 09:41:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/20 09:42:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/20 09:42:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


## Task 3 — Choose the best run from code and justify it

In [15]:
runs = mlflow.search_runs(
    experiment_names=[EXP],
    order_by=["metrics.test_rmse ASC"],
)

table = runs[["tags.mlflow.runName", "metrics.train_rmse",
              "metrics.test_rmse", "metrics.test_r2"]]
table.columns = ["run_name", "train_rmse", "test_rmse", "test_r2"]
print(table.round(4))

best_run_name = table.iloc[0]["run_name"]
best_run_id   = runs.iloc[0]["run_id"]
print("Best run:", best_run_name)

           run_name  train_rmse  test_rmse  test_r2
0  linear-base-0161      0.3465     0.3672   0.2808
1  ridge-a0.16-0161      0.3465     0.3672   0.2808
2   ridge-a1.6-0161      0.3465     0.3672   0.2807
3        rf-d6-0161      0.2715     0.3723   0.2609
Best run: linear-base-0161


**Why I chose this run (2–3 sentences):**


## Task 4 — Register, alias and roll back

In [16]:
# your code
from mlflow.tracking import MlflowClient
client = MlflowClient()

MODEL_NAME = f"LangganKu_Churn_{L4}"

base_run_id = runs.loc[runs["tags.mlflow.runName"] == f"linear-base-{L4}", "run_id"].iloc[0]

v1 = mlflow.register_model(f"runs:/{base_run_id}/model", MODEL_NAME)
v2 = mlflow.register_model(f"runs:/{best_run_id}/model", MODEL_NAME)
print(v1.version, v2.version)

Successfully registered model 'LangganKu_Churn_0161'.
2026/10/20 10:04:16 WARNING mlflow.tracking._model_registry.fluent: Run with id 169b5b971be64b26a1cae06b0ab9a586 has no artifacts at artifact path 'model', registering model based on models:/m-19800ef231404fe0870dd71e41facf5b instead


Created version '1' of model 'LangganKu_Churn_0161'.
Registered model 'LangganKu_Churn_0161' already exists. Creating a new version of this model...
2026/10/20 10:04:16 WARNING mlflow.tracking._model_registry.fluent: Run with id 169b5b971be64b26a1cae06b0ab9a586 has no artifacts at artifact path 'model', registering model based on models:/m-19800ef231404fe0870dd71e41facf5b instead


1 2


Created version '2' of model 'LangganKu_Churn_0161'.


In [17]:
client.set_registered_model_alias(MODEL_NAME, "champion", 1)
print("champion ->", client.get_model_version_by_alias(MODEL_NAME, "champion").version)

client.set_registered_model_alias(MODEL_NAME, "champion", 2)
print("champion ->", client.get_model_version_by_alias(MODEL_NAME, "champion").version)

client.set_model_version_tag(MODEL_NAME, "2", "approved_by", NAME)
client.update_model_version(MODEL_NAME, "2",
    description="Linear regression with default params; best test_rmse of the 3 candidates.")

champion -> 1
champion -> 2


<ModelVersion: aliases=['champion'], creation_timestamp=1792461856924, current_stage='None', deployment_job_state=None, description='Linear regression with default params; best test_rmse of the 3 candidates.', last_updated_timestamp=1792462007656, metrics=None, model_id=None, name='LangganKu_Churn_0161', params=None, run_id='169b5b971be64b26a1cae06b0ab9a586', run_link=None, source='models:/m-19800ef231404fe0870dd71e41facf5b', status='READY', status_message=None, tags={'approved_by': 'PETER LING HUA ZHONG'}, user_id=None, version=2, workspace='default'>

In [18]:
client.set_registered_model_alias(MODEL_NAME, "champion", 1)
print("after roll-back:", client.get_model_version_by_alias(MODEL_NAME, "champion").version)

client.set_registered_model_alias(MODEL_NAME, "champion", 2)
print("after roll-forward:", client.get_model_version_by_alias(MODEL_NAME, "champion").version)

after roll-back: 1
after roll-forward: 2


In [40]:
new = pd.DataFrame([
    {"tenure_months": 2, "support_calls": 4, "monthly_fee_rm": 55.0, "avg_hours_week": 1.5,
     "plan": "Premium", "payment_method": "Cash voucher", "contract": "Monthly", "main_device": "Phone"},
    {"tenure_months": 36, "support_calls": 0, "monthly_fee_rm": 25.0, "avg_hours_week": 15.0,
     "plan": "Basic", "payment_method": "Card", "contract": "Yearly", "main_device": "TV"},
    {"tenure_months": 14, "support_calls": 2, "monthly_fee_rm": 40.0, "avg_hours_week": 6.0,
     "plan": "Standard", "payment_method": "E-wallet", "contract": "6-month", "main_device": "Laptop"},
])



champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
print( champion.predict(new).tolist())

[0.9444478712605038, -0.4468873471598657, 0.2698883289408477]


In [ ]:
# mlflow ui --backend-store-uri sqlite:///langganku_0161.db

In [ ]:
# $env:MLFLOW_TRACKING_URI="sqlite:///langganku_0161.db"

In [ ]:
# mlflow models serve -m "models:/langganku_0161@champion" -p 5161 --env-manager local

## Task 5 — Serve the champion and call it

In [ ]:
# your code
import requests, json

URL = f"http://127.0.0.1:{PORT}/invocations"

def predict_via_api(df):
    payload = {"dataframe_split": json.loads(df.to_json(orient="split", index=False))}
    r = requests.post(URL, json=payload, timeout=60)
    preds = r.json().get("predictions") if r.status_code == 200 else None
    return r.status_code, preds

status, preds = predict_via_api(new_cars)
print(status, preds)